# Day 66：简历与作品集打磨：STAR 法则项目包装、GitHub 主页 README 与作品集生成器（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 10 周**：毕业答辩、简历作品集与面试冲刺结业  
> **GitLab 工单**：`STARMIND-066` ｜ **开发分支**：`feat/day66-resume-portfolio-kit`  
> **配套 HTML5 课件**：`courseware/day66/index.html` ｜ **独立版本快照**：`versions/day66/`

---

## 🎙️ 老陈开场旁白

早啊各位！很多同学技术学得很扎实，结果简历一投出去全石沉大海，为什么？因为 HR 和一面面试官扫一份简历平均只有 **15 秒**！如果你写'负责开发企业 RAG 知识库，使用 LangChain 和 FastAPI 完成接口开发'——这种流水账 100% 被直接筛掉！怎么写才能让面试官眼睛一亮？用 **STAR 法则 + 强动词 + 量化结果**：'针对纯向量检索对错误码召回率低（62%）痛点，主导设计 BM25+向量+RRF+Rerank 混合检索与置信度拒答机制，将 Ragas Context Recall 提升至 91%，幻觉率降至 3% 以下，并通过语义缓存与 7B 分级路由降低 54% Token 成本。' 今天咱们用代码写一个简历体检 Linter 和 GitHub 作品集生成器，把你的简历打磨到无可挑剔！

### 承上启下
- **昨日基线**：昨天我们高质量完成了星辰智脑毕业答辩，今天开始把这 65 天的硬核工程资产转化为能拿大厂面试邀约的王牌简历与 GitHub 作品集。
- **明日预告**：明天（Day 67）进入面试专题（一）——技术八股与手撕代码：手写 Attention、Top-P 采样、令牌桶限流与异步并发池。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day66'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-066）

学员即将投递国内互联网大厂及 AI 独角兽的 AI 应用工程师 / Agent 开发工程师 / LLM 后端岗位，需将 70 天项目成果转化为高转化率简历与可点击验收的 GitHub 作品集。

**今日迭代目标：**
- 实现 STAR 法则项目经历生成器与简历体检 Linter：自动检测弱动词、缺失量化数字与 ATS 关键词覆盖率。
- 实现 GitHub 作品集 README 生成器：自动拼装项目亮点、核心技术栈、量化 Benchmark 表格与快速启动命令。

**验收标准：**
- 简历体检器能准确识别'流水账弱描述'并给出具体扣分与整改建议，对 STAR 量化描述评定高分。
- GitHub README 生成器输出的 Markdown 包含技术徽章、架构模块清单、量化指标表与一键运行命令。
- 纯 Python 离线运行，pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. AI 应用工程师简历最加分的 5 个工程关键词

1) 混合检索 + Rerank + 引用溯源（证明懂 RAG 深水区）；2) 多 Agent + MCP + HITL 人工审批（证明懂企业安全边界）；3) FastAPI + SSE 流式推送（证明懂高并发异步工程）；4) Ragas 自动化评测 + 并发压测（证明懂质量闭环）；5) 语义缓存 + 分级路由 + 成本熔断（证明有 FinOps 老板视角）。

> **⚠️ 老陈避坑**：简历上写的每一个技术词都必须能在面试中讲出原理和踩坑经历，切忌'关键词堆砌但一问三不知'。

### 2. 简历里的量化数字必须经得起推敲

不要编造夸张的'千万级 QPS'或'准确率 99.99%'，真实、克制且有测试口径说明的数字（如'在 100 条企业内部黄金评测集上，Context Recall 由 68% 提升至 91%'）反而最让资深面试官信任。

> **⚠️ 老陈避坑**：写百分比提升时尽量给出基线（从 X% 提升至 Y%），只写'提升了 30%'显得缺乏严谨度。


## 💻 实操步骤 1：步骤一：实现 STAR 法则项目经历生成器与简历体检 Linter

- **目标文件**：`versions/day66/src/day66_star_resume_builder.py`
- **老陈代码旁白**：提供 STAR 结构化句式拼装器，并用 Linter 自动检查弱动词、量化指标与 ATS 关键词覆盖率。


In [ ]:
"""
Day 66 - 模块一：STAR 法则项目经历生成器与简历量化体检 Linter
"""
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Dict, List

STRONG_VERBS = ("主导", "设计", "架构", "构建", "实现", "重构", "优化", "落地", "搭建")
WEAK_VERBS = ("负责", "参与", "协助", "了解", "学习过", "使用过")
ATS_KEYWORDS = (
    "rag", "bm25", "rerank", "rrf", "agent", "langgraph", "mcp",
    "hitl", "fastapi", "sse", "vllm", "ragas", "docker", "token", "qps", "p95",
)


@dataclass
class StarEntry:
    situation_task: str
    action: str
    result: str

    def render(self) -> str:
        return f"针对{self.situation_task}，{self.action}，最终{self.result}。"


def lint_resume_bullet(bullet: str) -> Dict[str, object]:
    """对单条简历项目经历执行四维体检打分（满分 100）"""
    score = 0
    suggestions: List[str] = []
    low = bullet.lower()

    has_strong = any(v in bullet for v in STRONG_VERBS)
    has_weak = any(v in bullet for v in WEAK_VERBS)
    if has_strong and not has_weak:
        score += 25
    elif has_strong:
        score += 15
        suggestions.append("建议删除'负责/参与'等弱动词，直接用'主导/设计/落地'开头")
    else:
        suggestions.append("缺少强行动动词，建议以'主导设计/构建/优化'等动词突出个人贡献")

    # 检查量化数字（百分比、毫秒、倍数、金额、评分等）
    metrics = re.findall(r"\d+(?:\.\d+)?(?:%|ms|s|倍|元|条|万|k|qps)?", low)
    if len(metrics) >= 2:
        score += 35
    elif len(metrics) == 1:
        score += 20
        suggestions.append("仅有 1 处量化指标，建议补充优化前后对比基线（如从 X% 提升至 Y%）")
    else:
        suggestions.append("缺少量化成果数字！请补充 Recall/Faithfulness/P95延迟/Token降本等指标")

    matched_kw = [kw for kw in ATS_KEYWORDS if kw in low]
    if len(matched_kw) >= 3:
        score += 25
    elif matched_kw:
        score += 12
        suggestions.append("ATS 工程关键词偏少，建议补充具体技术方案词（如 BM25/Rerank/HITL/SSE/Ragas）")
    else:
        suggestions.append("未命中核心 AI 工程关键词，极易被简历初筛过滤")

    if 45 <= len(bullet) <= 160:
        score += 15
    else:
        suggestions.append(f"当前字数 {len(bullet)}，建议控制在 45~160 字以保证高信息密度")

    return {
        "score": score,
        "grade": "EXCELLENT" if score >= 85 else ("GOOD" if score >= 65 else "NEEDS_REWRITE"),
        "matched_keywords": matched_kw,
        "suggestions": suggestions,
    }


if __name__ == "__main__":
    star = StarEntry(
        situation_task="纯向量检索对企业专有错误码召回率偏低(62%)且偶发幻觉的痛点",
        action="主导设计 BM25+向量+RRF 混合检索与 Rerank 置信度拒答流水线，并落地 Ragas 自动化评测门禁",
        result="将 Context Recall 由 62% 提升至 91%，Faithfulness 达 92%，并通过语义缓存与 7B 分级路由降低 54% Token 成本",
    )
    bullet = star.render()
    print("生成 STAR 经历:", bullet)
    print("高分经历体检:", lint_resume_bullet(bullet))
    print("流水账体检:", lint_resume_bullet("负责参与公司知识库项目，使用了大模型和接口开发。"))


### 🔍 步骤 1 代码深度解析

把大厂面试官挑简历的隐性标准写成确定性 Linter：强动词（25分）+ 量化对比（35分）+ ATS 工程词（25分）+ 精炼度（15分），一眼定位差距。

**预期运行输出参考：**
```text
生成 STAR 经历: 针对纯向量检索对企业专有错误码召回率偏低(62%)...
高分经历体检: {'score': 100, 'grade': 'EXCELLENT', ...}
流水账体检: {'score': 0, 'grade': 'NEEDS_REWRITE', ...}
```


## 💻 实操步骤 2：步骤二：实现高转化率 GitHub 作品集 README 生成器

- **目标文件**：`versions/day66/src/day66_portfolio_generator.py`
- **老陈代码旁白**：根据项目元数据、技术栈与量化 Benchmark 指标，一键生成可直接放在 GitHub 主页的专业 Markdown README。


In [ ]:
"""
Day 66 - 模块二：高转化率 GitHub 作品集 README 自动生成器
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class ProjectPortfolioSpec:
    project_name: str
    tagline: str
    tech_stack: List[str]
    highlights: List[str]
    benchmarks: Dict[str, str]
    quickstart_cmd: str


def generate_github_readme(spec: ProjectPortfolioSpec) -> str:
    badges = " ".join(f"`{t}`" for t in spec.tech_stack)
    hl_lines = "\n".join(f"- {h}" for h in spec.highlights)
    bm_rows = "\n".join(f"| {k} | {v} |" for k, v in spec.benchmarks.items())
    return (
        f"# {spec.project_name}\n\n"
        f"> {spec.tagline}\n\n"
        f"**核心技术栈**: {badges}\n\n"
        "## 核心工程亮点\n"
        f"{hl_lines}\n\n"
        "## 量化评测与压测指标 (Benchmarks)\n"
        "| 评估维度 | 实测指标 |\n"
        "| --- | --- |\n"
        f"{bm_rows}\n\n"
        "## 一键本地运行与验收\n"
        "```bash\n"
        f"{spec.quickstart_cmd}\n"
        "```\n"
    )


if __name__ == "__main__":
    spec = ProjectPortfolioSpec(
        project_name="StarMind 星辰智脑 - 企业级 RAG & 多 Agent 平台",
        tagline="面向企业知识问答与高危审批自动化的生产级 AI Agent 全栈系统",
        tech_stack=["Python 3.11", "FastAPI", "SSE", "Hybrid RAG + Rerank", "Multi-Agent + HITL", "Docker"],
        highlights=[
            "混合检索四段式流水线（BM25 + 语义向量 + RRF + Rerank），支持强制 `[DOC-xx]` 溯源与低分拒答",
            "多 Agent 编排 + MCP 工具注册 + 金额超限 HITL 人工审批幂等闭环",
            "内置 Ragas 四维质量门禁、异步并发压测与语义缓存/7B 分级降本引擎",
        ],
        benchmarks={
            "Ragas Faithfulness (忠实度)": "92.4%",
            "Ragas Context Recall (召回率)": "91.0%",
            "并发压测 P95 延迟": "185ms (错误率 0.0%)",
            "Token 综合成本优化": "直降 54.0%",
        },
        quickstart_cmd="pytest -q && uvicorn src.day62_api_server:app --reload",
    )
    print(generate_github_readme(spec))


### 🔍 步骤 2 代码深度解析

生成的 README 兼顾可读性与硬核量化证据，面试官扫一眼就能看到你的架构能力与工程落地水准。

**预期运行输出参考：**
```text
# StarMind 星辰智脑 - 企业级 RAG & 多 Agent 平台
> 面向企业知识问答与高危审批自动化的生产级 AI Agent 全栈系统...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day66/tests/test_day66.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 66 单元测试：STAR 简历体检 Linter 与 GitHub 作品集生成器"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day66_star_resume_builder import StarEntry, lint_resume_bullet
from src.day66_portfolio_generator import ProjectPortfolioSpec, generate_github_readme


def test_star_builder_and_high_score_lint():
    entry = StarEntry(
        situation_task="企业知识库召回漏检且高频问答 Token 成本高的问题",
        action="主导设计 BM25+RRF+Rerank 混合检索与 FastAPI SSE 流式网关，引入 Ragas 评测与缓存路由",
        result="将召回率从 65% 提升至 91%，P95 延迟压至 180ms，月度 Token 成本降低 52%",
    )
    res = lint_resume_bullet(entry.render())
    assert res["grade"] == "EXCELLENT"
    assert res["score"] >= 85
    assert len(res["matched_keywords"]) >= 4


def test_weak_bullet_flagged_needs_rewrite():
    res = lint_resume_bullet("负责参与了公司的智能问答模块开发，使用了 Python。")
    assert res["grade"] == "NEEDS_REWRITE"
    assert len(res["suggestions"]) >= 3


def test_github_readme_contains_benchmarks_and_quickstart():
    md = generate_github_readme(
        ProjectPortfolioSpec(
            project_name="StarMind",
            tagline="企业级 AI Agent 平台",
            tech_stack=["FastAPI", "RAG"],
            highlights=["混合检索溯源"],
            benchmarks={"Faithfulness": "92%"},
            quickstart_cmd="pytest -q",
        )
    )
    assert "# StarMind" in md
    assert "| Faithfulness | 92% |" in md
    assert "pytest -q" in md

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：计算简历与目标岗位 JD 的关键词匹配度

**题目要求：**
编写 `calc_jd_match(resume_text, jd_keywords)` 返回命中率（0~1）与缺失关键词列表。

**思路提示：** 统一转小写后判断每个 `kw in resume_low`。


In [ ]:
def calc_jd_match(resume_text: str, jd_keywords: list[str]) -> dict:
    low = resume_text.lower()
    hit = [k for k in jd_keywords if k.lower() in low]
    miss = [k for k in jd_keywords if k.lower() not in low]
    return {"match_rate": round(len(hit) / max(1, len(jd_keywords)), 4), "missing": miss}


res = calc_jd_match("主导 RAG、FastAPI、SSE 与 Docker 落地", ["RAG", "FastAPI", "vLLM", "Docker"])
assert res["match_rate"] == 0.75 and res["missing"] == ["vLLM"]
print("JD 匹配度体检通过:", res)


**答案解析与评分标准：**

投递不同公司前按目标 JD 跑一次匹配度，查漏补缺，可显著提升简历初筛通过率。


### 作业 2（企业实战）：企业实战：批量体检整份简历并按得分从低到高输出待优化列表

**题目要求：**
对多条项目经历调用 `lint_resume_bullet`，返回按 `score` 升序排列的体检清单，优先改写最薄弱条目。

**思路提示：** 使用 `sorted(results, key=lambda x: x['score'])`。


In [ ]:
from src.day66_star_resume_builder import lint_resume_bullet

bullets = [
    "主导设计 BM25+Rerank+Ragas RAG 流水线，将召回率由 60% 提升至 90%，Token 成本降 50%。",
    "负责日常接口维护与文档整理。",
]
ranked = sorted([(b, lint_resume_bullet(b)["score"]) for b in bullets], key=lambda x: x[1])
assert ranked[0][1] < ranked[1][1]
print("最急需改写条目:", ranked[0])


**答案解析与评分标准：**

简历遵循木桶效应，把最薄弱的流水账条目改写成硬核 STAR 表述收益最大。


### 作业 3（架构进阶）：架构进阶：生成个人技术博客/作品集索引页（Portfolio Index JSON）

**题目要求：**
编写函数将多个项目按难度与主技术标签分组汇总为结构化字典，便于渲染个人求职主页。

**思路提示：** 遍历项目列表按 `category` 归类。


In [ ]:
def group_portfolio_projects(projects: list[dict]) -> dict[str, list[str]]:
    grouped: dict[str, list[str]] = {}
    for p in projects:
        grouped.setdefault(p["category"], []).append(p["name"])
    return grouped


out = group_portfolio_projects([
    {"name": "StarMind", "category": "Agent/RAG"},
    {"name": "LoRA-Eval-Gate", "category": "FineTuning"},
    {"name": "MCP-Gateway", "category": "Agent/RAG"},
])
assert out["Agent/RAG"] == ["StarMind", "MCP-Gateway"]
print("作品集分类索引通过:", out)


**答案解析与评分标准：**

主项目（星辰智脑）+ 2 个垂直小而美的工程组件，是最受大厂欢迎的作品集组合。


---

## 🎯 老陈收尾复盘

绝了！今天不仅把星辰智脑包装成了含金量拉满的 STAR 简历条，还一键生成了带架构徽章、Benchmark 量化表和一键启动指引的 GitHub README。明天起连打三天面试硬仗：八股手撕 -> 系统设计 -> 全真模拟面试！
